# Low-Complexity Sequence Analysis

    1. Deletion of PDB accessions
    
    2. Detection of Low-complexity sequences

    3. Compressing mapped data and storing in a parquet file for further analyisis

  In this notebook, firstly peptides with PDB accession numbers were identified and removed from the data. Secodly, LCRs were identified and excluded by applying a specific complexity threshold, ensuring higher data quality for downstream analysis. Finally, mapped data was compressed and peptide sequences that were not present in both human and virsus simultaneously were deleted. This final df were stored in a parquet file for further analysis.

In [1]:
import pandas as pd
import numpy as np
from collections import Counter

In [2]:
mapped = pd.read_parquet("data_sampled_10_percent.parquet")

In [3]:
mapped.head(20)

,peptide,side,accession,position
0,AAAAAAGSS,human,XP_054224810.1,66
1,AAAAAAGSS,human,XP_054224809.1,66
2,AAAAAAGSS,human,XP_054224808.1,66
3,AAAAAAGSS,human,XP_054224807.1,66
4,AAAAAAGSS,human,XP_054224806.1,66
5,AAAAAAGSS,human,XP_054224805.1,66
6,AAAAAAGSS,human,KAI4074409.1,66
7,AAAAAAGSS,human,KAI4022240.1,9
8,AAAAAAGSS,human,KAI2538467.1,9
9,AAAAAAGSS,human,AWQ13361.1,66


In [4]:
mapped.shape

(97744896, 4)

# Deletion of pdb Accessions

In [5]:
# Count number of rows that contains pdb accessions.
num_of_pdb = mapped.accession.str.contains("pdb").value_counts()

num_of_pdb

accession
False    97613958
True       130938
Name: count, dtype: int64

In [6]:
# Create a filtered dataframe that does not contain pdb accessions.
mapped_filtered = mapped[~mapped.accession.str.contains("pdb")]

mapped_filtered.head(20)

,peptide,side,accession,position
0,AAAAAAGSS,human,XP_054224810.1,66
1,AAAAAAGSS,human,XP_054224809.1,66
2,AAAAAAGSS,human,XP_054224808.1,66
3,AAAAAAGSS,human,XP_054224807.1,66
4,AAAAAAGSS,human,XP_054224806.1,66
5,AAAAAAGSS,human,XP_054224805.1,66
6,AAAAAAGSS,human,KAI4074409.1,66
7,AAAAAAGSS,human,KAI4022240.1,9
8,AAAAAAGSS,human,KAI2538467.1,9
9,AAAAAAGSS,human,AWQ13361.1,66


In [7]:
mapped_filtered.shape

(97613958, 4)

 PDB accession numbers were deleted because they were already present in the data along with their NCBI accession numbers, and most of the proteins with PDB accession numbers were mutated and belonged to artificial proteins.

# Detection of Low-complexity Sequences

Low Complexity Regions (LCRs), or compositionally biased regions, are protein segments with low amino acid diversity. During homology searches, this lack of diversity can lead to artificially high similarity scores. Since these regions are often biologically uninformative for sequence alignment, these high scores can be misleading. Therefore, penalizing LCRs to avoid bias in homology searches is vital.

In [8]:
def is_low_complexity(peptide):
        counts = Counter(peptide)
        max_count = max(counts.values())
    
        lcr_score = max_count / 9
        n_unique = int(len(counts))
        
        return (lcr_score >= 5/9) | (n_unique <=3)  

mapped_filtered['is_low_complexity'] = mapped_filtered['peptide'].apply(is_low_complexity) 

/var/folders/2v/z3sjntt94sx410m__1f2pfz00000gn/T/ipykernel_6596/2402096183.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mapped_filtered['is_low_complexity'] = mapped_filtered['peptide'].apply(is_low_complexity)


 The complexity threshold was set at 5/9. This criteria classifies a peptide as low-complexity if a single amino acid repeats five or more times within the 9-mer sequence. Additionally, any sequence containing three or fewer unique amino acids is also flagged as low-complexity.

 Within the mapped dataset, all protein sequences are divided into overlapping k-mers using a window size of 9. Therefore, the k-mer length was treated as a constant in all scoring calculations across the analysis.

In [9]:
mapped_filtered["is_low_complexity"].value_counts()

is_low_complexity
False    97379507
True       234451
Name: count, dtype: int64

234,451 peptides were identified as having low complexity regions.

In [10]:
# Create a new df without True values (Low-complexity).
mapped_df = mapped_filtered[~mapped_filtered["is_low_complexity"] == True]

In [11]:
mapped_df.head(10)

,peptide,side,accession,position,is_low_complexity
2048,AAAYYVGYL,virus,UWK54271.1,264,False
2049,AAAYYVGYL,virus,UWK54259.1,264,False
2050,AAAYYVGYL,virus,UWK54235.1,264,False
2051,AAAYYVGYL,virus,UWL26814.1,261,False
2052,AAAYYVGYL,virus,UWL26743.1,261,False
2053,AAAYYVGYL,virus,UWK54188.1,264,False
2054,AAAYYVGYL,virus,UWL26708.1,261,False
2055,AAAYYVGYL,virus,UWK54153.1,264,False
2056,AAAYYVGYL,virus,UWK54141.1,264,False
2057,AAAYYVGYL,virus,UOM26858.1,264,False


# Compressing Mapped Data and Storing in a Parquet File for Further Analysis

In [12]:
mapped_filtered.shape

(97613958, 5)

In [13]:
mapped_filtered.head(5)

,peptide,side,accession,position,is_low_complexity
0,AAAAAAGSS,human,XP_054224810.1,66,True
1,AAAAAAGSS,human,XP_054224809.1,66,True
2,AAAAAAGSS,human,XP_054224808.1,66,True
3,AAAAAAGSS,human,XP_054224807.1,66,True
4,AAAAAAGSS,human,XP_054224806.1,66,True


In [14]:
compr_mapped = pd.pivot_table(mapped_filtered, index = "peptide", columns = "side", values = ["accession","position"], aggfunc = lambda x: ",".join(map(str,x)))

compr_mapped.columns = [f"{side}_{attr}"for attr, side in compr_mapped.columns]

compr_mapped = compr_mapped.reset_index().fillna("")

compr_mapped['is_low_complexity'] = compr_mapped['peptide'].apply(is_low_complexity)

In [15]:
compr_mapped.shape

(16069, 6)

To improve data loading speed and processing efficiency, the mapped_filtered data frame (not included pdb accessions), which previously contained 97,613,958 rows, has been compressed and now contains 16,069 rows.

In [16]:
compr_mapped.head(5)

,peptide,human_accession,virus_accession,human_position,virus_position,is_low_complexity
0,AAAAAAGSS,"XP_054224810.1,XP_054224809.1,XP_054224808.1,X...",,"66,66,66,66,66,66,66,9,9,66",,True
1,AAAAAAGTA,EAX06575.1,"AGK27098.1,XSG61943.1,ALM55118.1,AFD22136.1,AF...",354,"7,187,7,7,7",True
2,AAAAAAGTG,,"XPO04980.1,YP_003969768.1",,"122,142",True
3,AAAAAAGVG,NP_005802.1,"WWY65371.1,WNT47152.1,YP_009784297.1,YP_009789...",40,"115,1351,127,127",True
4,AAAAAAGVL,"AAY34147.1,XP_047293496.1","XOQ95597.1,XOR21395.1","183,183","98,200",True


In [17]:
mapped_human_virus_df = compr_mapped[~((compr_mapped["virus_accession"] == "") | (compr_mapped["human_accession"] == ""))]

For any peptide, if there was no value for either the virus or humans, those rows were deleted. Thus, a data frame was created containing all matching peptides in humans and viruses, enabling the understanding of potential molecular mimicry.

In [18]:
mapped_human_virus_df.shape

(11768, 6)

Due to the deletion of these rows, the number of data rows has decreased to 11,768.

In [19]:
mapped_human_virus_df.to_parquet("compressed_mapped.parquet", index = False)

Subsequent operations will be performed using the compressed_mapped.parquet file.